# Config

In [1]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [51]:
import pandas as pd
import os
from preprocess.preprocess import clean_text
from preprocess.translate import translator, gen_text_for_embedding
import time

# 1) Preprocesamiento de los datos


In [141]:
# 1) Cargar datos
path = "/tmp/data"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")

# 2) Preprocesar los datos

#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)

# 2) Traducción del texto

## Translator

In [69]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import MarianMTModel, MarianTokenizer
import torch
import langid
import re
import pandas as pd
from tqdm import tqdm
import torch
import numpy as np

def gen_ids(num, list):
    """
    Repite el ID tantas veces como elementos haya en la lista.
    Args:
        num (int): ID a repetir.
        list (list): Lista cuyos elementos determinan cuántas veces repetir el ID.
    Returns:
        list: Lista con el ID repetido.
    """
    return [num] * len(list)
    
def detect_language(texts):
    """
    Detecta si los textos están en español.
    Args:
        texts (list): Lista de textos a evaluar.
    Returns:
        list: Lista de booleanos indicando si cada texto está en español.
    """
    langs =[]
    for text in texts:
        lang, _ = langid.classify(text)
        if lang == 'es':
            langs.append(True)
        else:
            langs.append(False)
    return langs

class translator():
    """
    Clase para traducir texto del español al inglés utilizando un modelo y tokenizer de Hugging Face.
    Incluye detección de idioma, segmentación en fragmentos y unión de la traducción final.
    """
    def __init__(self, model, tokenizer, max_input_tokens=512):
        """
        Inicializa el traductor cargando el modelo y tokenizer en GPU si está disponible.
        Args:
            model: Modelo de traducción.
            tokenizer: Tokenizer asociado al modelo.
            max_input_tokens (int): Máximo de tokens por fragmento.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Usando dispositivo: {self.device}")
        # Cargar modelo y tokenizer
        self.tokenizer = tokenizer
        self.model = model.to(self.device)
        self.max_input_tokens = max_input_tokens

    def split_text(self, text_to_split):
        """
        Divide un texto en fragmentos manejables según el límite de tokens del modelo.
        Args:
            text_to_split (str): Texto original a dividir.
        Returns:
            list: Lista de fragmentos como objetos Document.
        """
        # Splitter basado en el tokenizador de Helsinki (cuenta tokens reales)
        text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            tokenizer=self.tokenizer,
            chunk_size=self.max_input_tokens,
            chunk_overlap=0,
            separators=["\n\n", ".", ",", " "] #Jerarquía de separadores
        )

        texts = text_splitter.create_documents([text_to_split])
        return texts

    def parallel_translate(self, texts, batch_size=16):
        """
        Traduce una lista de textos en paralelo.
        Args:
            texts (list): Lista de textos a traducir.
            batch_size (int): Tamaño del lote para la traducción.
        Returns:
            list: Lista de textos traducidos.
        """
        #Crear lista de id
        id = list(range(len(texts)))
        # Identifica textos en español
        langs = detect_language(texts)

        #Crea un dataframe con id, langs y texto
        df = pd.DataFrame({'id': id, 'is_spanish': langs, 'text': texts})
        #Divide el dataframe en dos: textos en español y textos en otros idiomas
        df_spanish = df[df['is_spanish']]
        df_other = df[~df['is_spanish']]
        #Realiza split de los textos en español
        df_spanish['splits'] = df_spanish['text'].apply(lambda t: self.split_text(t))
        #Concatenación de splits con ids para no perder referencia
        
        return df_spanish, df_other

In [70]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Usando dispositivo: cuda


In [ ]:
#df_test = df.iloc[:64]  # Subset para pruebas rápidas

In [ ]:
texts = df_test['Resumen_trad'].tolist()
#Crear lista de id
id = list(range(len(texts)))
# Identifica textos en español
langs = detect_language(texts)

#Crea un dataframe con id, langs y texto
df = pd.DataFrame({'id': id, 'is_spanish': langs, 'text': texts})
#Divide el dataframe en dos: textos en español y textos en otros idiomas
df_spanish = df[df['is_spanish']]
df_other = df[~df['is_spanish']]
#Realiza split de los textos en español
df_spanish['text'] = df_spanish['text'].apply(lambda t: trans.split_text(t))
#Concatenación de splits con ids para no perder referencia
id_loc = []
texts_for_batch = []
for index, row in df_spanish.iterrows():
    id_loc.extend(gen_ids(row['id'], row['text']))
    texts_for_batch.extend(row['text'])

In [105]:
texts = texts_for_batch
translated_chunks = []
batch_size=16

for i in range(0, len(texts), batch_size):
    batch = [t.page_content for t in texts[i:i+batch_size]]
    print(len(batch))
    encoded = trans.tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=512).to(trans.device)
    with torch.inference_mode():
        out_ids = trans.model.generate(
            **encoded,
            num_beams=4,
            max_new_tokens=trans.max_input_tokens,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    translated_batch = trans.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
    translated_chunks.extend(translated_batch)

16
16
16
16
16
16
12


In [112]:
#Reagrupar textos utilizando id_loc
df_translated = pd.DataFrame({
    'id': id_loc,
    'text': translated_chunks
})
# Agrupar por id y unir textos traducidos
df_translated = df_translated.groupby('id')['text'].apply(lambda x: ' '.join(x)).reset_index()

In [116]:
#Unir textos traducidos con los textos originales
df_final = pd.concat([df_translated, df_other.drop(columns=['is_spanish'])], ignore_index=True)
df_final = df_final.sort_values(by="id").reset_index(drop=True)
df_final.shape

(64, 2)

In [114]:
df_final.head(10)

,id,text
0,0,general objectives: to describe the processes ...
1,1,In order to evaluate the behaviors related to ...
2,2,the present project involves the realization o...
3,3,the study of the motivation has different side...
4,4,secondary consolidation and ageing are two oft...
5,5,thermal lens spectroscopy (tls) had shown to b...
6,6,at the national and international level is rec...
7,7,iv. summary of the project () the silvopastora...
8,8,"any historiographical, research and compilatio..."
9,9,the proposal is situated in the field of quali...


# Ordenar codigo

In [ ]:
texts = df_test['Resumen_trad'].tolist()
#Crear lista de id
id = list(range(len(texts)))
# Identifica textos en español
langs = detect_language(texts)

#Crea un dataframe con id, langs y texto
df = pd.DataFrame({'id': id, 'is_spanish': langs, 'text': texts})
#Divide el dataframe en dos: textos en español y textos en otros idiomas
df_spanish = df[df['is_spanish']]
df_other = df[~df['is_spanish']]
#Realiza split de los textos en español (split evita pasarse del largo máximo de tokens de entrada)
df_spanish['text'] = df_spanish['text'].apply(lambda t: trans.split_text(t))
#Concatenación de splits con ids para no perder referencia
id_loc = []
texts_for_batch = []
for index, row in df_spanish.iterrows():
    id_loc.extend(gen_ids(row['id'], row['text']))
    texts_for_batch.extend(row['text'])
#Traducción de textos
texts = texts_for_batch
translated_chunks = []
batch_size=16
for i in range(0, len(texts), batch_size):
    batch = [t.page_content for t in texts[i:i+batch_size]]
    print(len(batch))
    encoded = trans.tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=512).to(trans.device)
    with torch.inference_mode():
        out_ids = trans.model.generate(
            **encoded,
            num_beams=4,
            max_new_tokens=trans.max_input_tokens,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    translated_batch = trans.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
    translated_chunks.extend(translated_batch)

#Reagrupar textos utilizando id_loc
df_translated = pd.DataFrame({
    'id': id_loc,
    'text': translated_chunks
})
# Agrupar por id y unir textos traducidos
df_translated = df_translated.groupby('id')['text'].apply(lambda x: ' '.join(x)).reset_index()

#Unir textos traducidos con los textos originalmente en inglés
df_translated = pd.concat([df_translated, df_other.drop(columns=['is_spanish'])], ignore_index=True)
df_translated = df_translated.sort_values(by="id").reset_index(drop=True)
df_translated.shape

# Convertir en clase

In [122]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import MarianMTModel, MarianTokenizer
import torch
import langid
import re
import pandas as pd
from tqdm import tqdm
import torch

def gen_ids(num, list):
    """
    Repite el ID tantas veces como elementos haya en la lista.
    Args:
        num (int): ID a repetir.
        list (list): Lista cuyos elementos determinan cuántas veces repetir el ID.
    Returns:
        list: Lista con el ID repetido.
    """
    return [num] * len(list)
    
def detect_language(texts):
    """
    Detecta si los textos están en español.
    Args:
        texts (list): Lista de textos a evaluar.
    Returns:
        list: Lista de booleanos indicando si cada texto está en español.
    """
    langs =[]
    for text in texts:
        lang, _ = langid.classify(text)
        if lang == 'es':
            langs.append(True)
        else:
            langs.append(False)
    return langs


class translator():
    """
    Clase para traducir texto del español al inglés utilizando un modelo y tokenizer de Hugging Face.
    Incluye detección de idioma, segmentación en fragmentos y unión de la traducción final.
    """
    def __init__(self, model, tokenizer, max_input_tokens=512):
        """
        Inicializa el traductor cargando el modelo y tokenizer en GPU si está disponible.
        Args:
            model: Modelo de traducción.
            tokenizer: Tokenizer asociado al modelo.
            max_input_tokens (int): Máximo de tokens por fragmento.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Usando dispositivo: {self.device}")
        # Cargar modelo y tokenizer
        self.tokenizer = tokenizer
        self.model = model.to(self.device)
        self.max_input_tokens = max_input_tokens

    def split_text(self, text_to_split):
        """
        Divide un texto en fragmentos manejables según el límite de tokens del modelo.
        Args:
            text_to_split (str): Texto original a dividir.
        Returns:
            list: Lista de fragmentos como objetos Document.
        """
        # Splitter basado en el tokenizador de Helsinki (cuenta tokens reales)
        text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            tokenizer=self.tokenizer,
            chunk_size=self.max_input_tokens,
            chunk_overlap=0,
            separators=["\n\n", ".", ",", " "] #Jerarquía de separadores
        )

        texts = text_splitter.create_documents([text_to_split])
        return texts

    def translate_esp_en(self, text_to_split, batch_size=16):
        texts = self.split_text(text_to_split)
        translated_chunks = []

        for i in range(0, len(texts), batch_size):
            batch = [t.page_content for t in texts[i:i+batch_size]]
            encoded = self.tokenizer(batch, return_tensors="pt", padding=True,
                                    truncation=True, max_length=512).to(self.device)
            with torch.inference_mode():
                out_ids = self.model.generate(
                    **encoded,
                    num_beams=4,
                    max_new_tokens=self.max_input_tokens,
                    no_repeat_ngram_size=3,
                    early_stopping=True
                )
            translated_batch = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
            translated_chunks.extend(translated_batch)

        return "\n\n".join(translated_chunks)

    # Detección y traducción de texto. 
    def detect_and_translate(self, text):
        """
        Detecta el idioma del texto y lo traduce si está en español.
        Args:
            text (str): Texto de entrada.
        Returns:
            str: Texto traducido o el mismo texto si no es español.
        """
        lang, _ = langid.classify(text)
        if lang == 'es':
            return self.translate_esp_en(text)
        
        return text

    def translate_parallel(self, texts):
        #Crear lista de id
        id = list(range(len(texts)))
        # Identifica textos en español
        langs = detect_language(texts)

        #Crea un dataframe con id, langs y texto
        df = pd.DataFrame({'id': id, 'is_spanish': langs, 'text': texts})
        #Divide el dataframe en dos: textos en español y textos en otros idiomas
        df_spanish = df[df['is_spanish']]
        df_other = df[~df['is_spanish']]
        #Realiza split de los textos en español (split evita pasarse del largo máximo de tokens de entrada)
        df_spanish['text'] = df_spanish['text'].apply(lambda t: self.split_text(t))
        #Concatenación de splits con ids para no perder referencia
        id_loc = []
        texts_for_batch = []
        for index, row in df_spanish.iterrows():
            id_loc.extend(gen_ids(row['id'], row['text']))
            texts_for_batch.extend(row['text'])

        #Traducción de textos
        texts = texts_for_batch
        translated_chunks = []
        batch_size=16
        for i in range(0, len(texts), batch_size):
            batch = [t.page_content for t in texts[i:i+batch_size]]
            print(len(batch))
            encoded = self.tokenizer(batch, return_tensors="pt", padding=True,
                                    truncation=True, max_length=512).to(self.device)
            with torch.inference_mode():
                out_ids = self.model.generate(
                    **encoded,
                    num_beams=4,
                    max_new_tokens=self.max_input_tokens,
                    no_repeat_ngram_size=3,
                    early_stopping=True
                )
            translated_batch = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
            translated_chunks.extend(translated_batch)

        #Reagrupar textos utilizando id_loc
        df_translated = pd.DataFrame({
            'id': id_loc,
            'text': translated_chunks
        })
        # Agrupar por id y unir textos traducidos
        df_translated = df_translated.groupby('id')['text'].apply(lambda x: ' '.join(x)).reset_index()

        #Unir textos traducidos con los textos originalmente en inglés
        df_translated = pd.concat([df_translated, df_other.drop(columns=['is_spanish'])], ignore_index=True)
        df_translated = df_translated.sort_values(by="id").reset_index(drop=True)    

        return df_translated["text"].to_list()



# Convertir en clase 2

In [165]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import MarianMTModel, MarianTokenizer
import torch
import langid
import re
import pandas as pd
from tqdm import tqdm
import torch

def gen_ids(num, list):
    """
    Repite el ID tantas veces como elementos haya en la lista.
    Args:
        num (int): ID a repetir.
        list (list): Lista cuyos elementos determinan cuántas veces repetir el ID.
    Returns:
        list: Lista con el ID repetido.
    """
    return [num] * len(list)
    
def detect_language(texts):
    """
    Detecta si los textos están en español.
    Args:
        texts (list): Lista de textos a evaluar.
    Returns:
        list: Lista de booleanos indicando si cada texto está en español.
    """
    langs =[]
    for text in texts:
        lang, _ = langid.classify(text)
        if lang == 'es':
            langs.append(True)
        else:
            langs.append(False)
    return langs


class translator():
    """
    Clase para traducir texto del español al inglés utilizando un modelo y tokenizer de Hugging Face.
    Incluye detección de idioma, segmentación en fragmentos y unión de la traducción final.
    """
    def __init__(self, model, tokenizer, max_input_tokens=512):
        """
        Inicializa el traductor cargando el modelo y tokenizer en GPU si está disponible.
        Args:
            model: Modelo de traducción.
            tokenizer: Tokenizer asociado al modelo.
            max_input_tokens (int): Máximo de tokens por fragmento.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Usando dispositivo: {self.device}")
        # Cargar modelo y tokenizer
        self.tokenizer = tokenizer
        self.model = model.to(self.device)
        self.max_input_tokens = max_input_tokens

    def split_text(self, text_to_split):
        """
        Divide un texto en fragmentos manejables según el límite de tokens del modelo.
        Args:
            text_to_split (str): Texto original a dividir.
        Returns:
            list: Lista de fragmentos como objetos Document.
        """
        # Splitter basado en el tokenizador de Helsinki (cuenta tokens reales)
        text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            tokenizer=self.tokenizer,
            chunk_size=self.max_input_tokens,
            chunk_overlap=0,
            separators=["\n\n", ".", ",", " "] #Jerarquía de separadores
        )

        texts = text_splitter.create_documents([text_to_split])
        return texts

    def translate_esp_en(self, text_to_split, batch_size=16):
        texts = self.split_text(text_to_split)
        translated_chunks = []

        for i in range(0, len(texts), batch_size):
            batch = [t.page_content for t in texts[i:i+batch_size]]
            encoded = self.tokenizer(batch, return_tensors="pt", padding=True,
                                    truncation=True, max_length=512).to(self.device)
            with torch.inference_mode():
                out_ids = self.model.generate(
                    **encoded,
                    num_beams=4,
                    max_new_tokens=self.max_input_tokens,
                    no_repeat_ngram_size=3,
                    early_stopping=True
                )
            translated_batch = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
            translated_chunks.extend(translated_batch)

        return "\n\n".join(translated_chunks)

    # Detección y traducción de texto. 
    def detect_and_translate(self, text):
        """
        Detecta el idioma del texto y lo traduce si está en español.
        Args:
            text (str): Texto de entrada.
        Returns:
            str: Texto traducido o el mismo texto si no es español.
        """
        lang, _ = langid.classify(text)
        if lang == 'es':
            return self.translate_esp_en(text)
        
        return text

    def translate_parallel(self, texts, batch_size=16):
        """
        Traduce en paralelo una lista de textos mezclados en español e inglés.

        Inputs:
            texts (list[str]): Lista de textos a traducir.

        Outputs:
            list[str]: Lista de textos donde los que estaban en español fueron traducidos
                    y los que estaban en otros idiomas se mantienen igual.

        Proceso:
            1. Detecta qué textos están en español.
            2. Divide los textos largos en fragmentos (para no superar límite de tokens).
            3. Traduce por lotes con el modelo de traducción.
            4. Reconstruye los textos traducidos completos.
            5. Une los textos traducidos con los originales en otros idiomas, manteniendo orden.
        """

        # Crear lista de IDs únicos para no perder el orden
        ids = list(range(len(texts)))

        # Detectar idioma de cada texto (True = español, False = otro idioma)
        langs = detect_language(texts)
   

        # Construir dataframe base
        df = pd.DataFrame({'id': ids, 'is_spanish': langs, 'text': texts})

        # Separar en textos español e inglés
        df_spanish = df[df['is_spanish']].copy()
        df_other   = df[~df['is_spanish']].copy()

        # Dividir textos españoles en fragmentos manejables
        df_spanish['text'] = df_spanish['text'].apply(self.split_text)

        # Generar lista expandida de (id, fragmento)
        id_loc, texts_for_batch = [], []
        for _, row in df_spanish.iterrows():
            id_loc.extend(gen_ids(row['id'], row['text']))   # genera ID por fragmento
            texts_for_batch.extend(row['text'])              # agrega fragmentos

        # === Traducción por lotes ===
        translated_chunks = []
        
        for i in tqdm(range(0, len(texts_for_batch), batch_size), desc="Traduciendo", unit="batch"):
            batch = [t.page_content for t in texts_for_batch[i:i+batch_size]]
            
            # Tokenizar batch
            encoded = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(self.device)

            # Generación de traducción
            with torch.inference_mode():
                out_ids = self.model.generate(
                    **encoded,
                    num_beams=4,
                    max_new_tokens=self.max_input_tokens,
                    no_repeat_ngram_size=3,
                    early_stopping=True
                )
            # Decodificar traducciones y acumular
            translated_batch = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
            translated_chunks.extend(translated_batch)

        # === Reconstruir textos ===
        df_translated = pd.DataFrame({'id': id_loc, 'text': translated_chunks})
        df_translated = (
            df_translated.groupby('id')['text']
            .apply(lambda x: ' '.join(x))      # unir fragmentos del mismo texto
            .reset_index()
        )

        # Unir con textos originales en otros idiomas y ordenar
        df_final = pd.concat(
            [df_translated, df_other.drop(columns=['is_spanish'])],
            ignore_index=True
        ).sort_values(by="id").reset_index(drop=True)

        return df_final["text"].to_list()


In [171]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

df_test["Resumen_trad_test"] = trans.translate_parallel(df_test["Resumen_trad"].to_list(), batch_size=8)

/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Usando dispositivo: cuda


Traduciendo: 100%|██████████| 7/7 [00:37<00:00,  5.41s/batch]
/tmp/ipykernel_778/1519796376.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["Resumen_trad_test"] = trans.translate_parallel(df_test["Resumen_trad"].to_list(), batch_size=8)


In [162]:
df_test["Resumen_trad_test"] = df_test["Resumen_trad"].apply(trans.detect_and_translate)

/tmp/ipykernel_778/788704901.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["Resumen_trad_test"] = df_test["Resumen_trad"].apply(trans.detect_and_translate)


In [172]:
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

for src, dst in cols.items():
    df_test[dst] = trans.translate_parallel(df_test[src].to_list(), batch_size=8)


Traduciendo: 100%|██████████| 4/4 [00:02<00:00,  1.85batch/s]
/tmp/ipykernel_778/3337656691.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test[dst] = trans.translate_parallel(df_test[src].to_list(), batch_size=8)
Traduciendo: 100%|██████████| 7/7 [00:38<00:00,  5.44s/batch]
/tmp/ipykernel_778/3337656691.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test[dst] = trans.translate_parallel(df_test[src].to_list(), batch_size=8)
Traduciendo: 100%|██████████| 2/2 [00:00<00:00,  3.02batch/s]
/tmp